In [1]:
import pandas as pd
import numpy as np
from scipy.stats import norm
from scipy.optimize import brentq

# 1. 載入原始資料與參數設定
# 假設原始資料路徑為 'TXO_2019_Raw_Data.csv'
# df_raw = pd.read_csv('TXO_2019_Raw_Data.csv')
r = 0.0105  # 無風險利率，需與組員一致 [cite: 45, 67]

# 2. 資料清理與初步篩選 [cite: 39, 41, 44]
def clean_txo_data(df):
    # 僅保留台指選擇權 (TXO) 並篩選成交量 > 0 [cite: 41, 43]
    df = df[df['Product_Code'] == 'TXO'].copy()
    df = df[df['Volume'] > 0]

    # 定義看漲 (Call) 與看跌 (Put) 投資人 [cite: 39, 70]
    # 假設 Contract 欄位包含 'C' 或 'P'
    df['Type'] = df['Contract'].apply(lambda x: 'Call' if 'C' in x else 'Put')

    # 處理日期與 Maturity (T) [cite: 45, 67]
    df['Date'] = pd.to_datetime(df['Date'])
    # T = (到期日 - 交易日) / 365
    return df

# 3. 隱含波動率 (IV) 計算模型 (BS Model + 數值方法) [cite: 40, 68]
def bs_iv_solver(price, S, K, T, r, option_type):
    def objective_function(sigma):
        d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
        d2 = d1 - sigma * np.sqrt(T)
        if option_type == 'Call':
            res = S * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)
        else:
            res = K * np.exp(-r * T) * norm.cdf(-d2) - S * norm.cdf(-d1)
        return res - price

    try:
        # 使用 brentq (數值方法) 反推 IV [cite: 40]
        return brentq(objective_function, 0.0001, 5.0)
    except:
        return np.nan # 紀錄無法反推出 IV 之樣本

# 4. 每日統計量運算 (Mean, Std, PCR) [cite: 42, 43]
# 分別計算看漲與看跌投資人之 IV 平均數與「樣本」標準差 [cite: 19, 27, 42]
def calculate_daily_metrics(df):
    # 計算每日 IV 統計
    daily_stats = df.groupby(['Date', 'Type'])['IV'].agg(['mean', 'std']).unstack()

    # 計算 PCR (Put/Call Ratio) [cite: 20, 43, 69]
    volume_stats = df.groupby(['Date', 'Type'])['Volume'].sum().unstack()
    daily_stats['PCR'] = volume_stats['Put'] / volume_stats['Call']

    return daily_stats

# 5. 匯出結果 (產出您的 CSV 檔案) [cite: 49, 78]
# final_df.to_csv('2019_TXO_Final_Analysis_Fixed_V3.csv')